# Scientific Python
## Central European University

## 05 Error handling, JSON, XML, web scraping -- exercises

Instructor: Márton Pósfai

Email: posfaim@ceu.edu

*Don't forget:* use the Slack channel for discussion, to ask questions, or to show solutions to exercises that are different from the ones provided in the notebook. [Slack channel](http://www.personal.ceu.edu/staff/Marton_Posfai/slack_forward.html)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import urllib
from bs4 import BeautifulSoup
import datetime

### Final problem of last class

Your submission will be graded based on correctness and simplicity. This does not mean that your code has to be the shortest possible, but that you should avoid unnecessary steps.

You can find the results of the guessing game on moodle in a file containing four columns: the name of the guesser, the index of the question, the guess, and the academic term the guess was made. Download the file and load it as a dataframe. Do the following steps:
* Clean and pre-process the data:
    * The guesses should be integers but the form allowed any entry. Convert the guesses to integers, if it cannot be converted drop the row (check out `pd.to_numeric()`).
    * Some internet trolls might tried to wreak havoc by submitting clearly wrong guesses. Drop any row that has a guess larger than 100,000 or less than 0.
* Print out the mean guess for each image.
* Print out the mean guess for each image but only considering guesses made this term.
* The correct answers to the questions will be/already are posted on slack. Create a new column named 'dist_from_correct' in the dataframe containing the difference between the guess and the correct value. Pick one question and plot a histogram of this new column and add a vertical line corresponding to the group average guess (check out `plt.axvline()`).

In [ ]:
Dcorrect = {'Q1_green_squares':476,
             'Q2_red_circles':893,
             'Q3_black_triangles':568,
             'Q4_blue_crosses':2069,
             'Q5_purple_circles':312}

In [ ]:
term = "2024_25_winter"

In [ ]:
df = pd.read_csv('guessing_game_images/guesses.csv')

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df[df['Name']=='Mr. and Mrs. Troll']

In [ ]:
df['Guess']=pd.to_numeric(df.Guess, errors='coerce',downcast='integer')
df=df.dropna()

In [ ]:
df[df.Guess==3.14]

In [ ]:
df.Guess = df.Guess.astype(int)
df[df.Guess==3]

In [ ]:
#df.drop(df[(df.Guess < 0) | (df.Guess > 100000)].index, inplace=True)
df = df[(df.Guess >= 0) & (df.Guess < 100000)]

In [ ]:
df["Guess"] = df["Guess"].apply(lambda x: x if x%1 == 0 else np.nan)
df[df['Name']=='Mr. and Mrs. Troll']

In [ ]:
df.dropna(inplace=True)

In [ ]:
Gmeans = df.groupby('QuestionID')['Guess'].mean()
Gmeans

In [ ]:
GmeansThisTerm = df[df['Term']==term].groupby('QuestionID')['Guess'].mean()
GmeansThisTerm

In [ ]:
df['diff']=df.apply(lambda row: row['Guess']-Dcorrect[row['QuestionID']],axis=1)

In [ ]:
df

In [ ]:
print(df.groupby('QuestionID')['diff'].max())

In [ ]:
axs=df['diff'].hist(by=df['QuestionID'],figsize=(14,10),bins=30);
diff_means = df.groupby('QuestionID')['diff'].mean()
for ax,qid in zip(axs.reshape(-1),Dcorrect.keys()):
    ax.axvline(0,color='r')
    ax.axvline(diff_means[qid],color='k')

Who made the best guesses?

In [ ]:
df['reldiff']=df.apply(
    lambda row: np.abs(row['Guess']-Dcorrect[row['QuestionID']])/Dcorrect[row['QuestionID']],
    axis=1)
df.loc[ df.groupby('QuestionID')['reldiff'].idxmin() ]

In [ ]:
df.loc[df[df['Term']==term].groupby('QuestionID')['reldiff'].idxmin()]

In [ ]:
df.groupby('QuestionID')['reldiff'].min()

In [ ]:
for qid,value in df.groupby('QuestionID')['reldiff'].min().items():
    guessers = df['Name'][(df['reldiff']==value) & (df['QuestionID']==qid)]
    print(qid,':', guessers.values)

Which term was the best?

In [ ]:
byterm_df = df.groupby(['QuestionID','Term'])['diff'].agg(['mean','count'])
byterm_df.columns = ['diff','count']
byterm_df.reset_index(inplace=True)
byterm_df

In [ ]:
byterm_df['reldiff']=byterm_df.apply(
    lambda row: np.abs(row['diff'])/Dcorrect[row['QuestionID']],
    axis=1)

In [ ]:
reldiff_df = byterm_df.pivot(index='Term', columns='QuestionID',values='reldiff')

In [ ]:
reldiff_df

In [ ]:
reldiff_df.plot(kind='bar')
plt.ylabel('Relative difference of mean');

## Registering for random webapi-s (and other websites)

Consider using a temporary email address like:
https://www.google.com/search?q=temporary+email

Some sites might block some of the temp email providers. In this case, just try the next one.


## Exercises -- Error handling

### 01 Dictionaries and errors

Write a function that does the same thing as the `get` method of dictionaries:
* takes a dictionary and a key as input
* if the key exists, return the corresponding value
* if the key does not exist, return `None`

Use the `try`-`except` pair!

<details><summary><u>Hint.</u></summary>
<p>

If you try to access a key that doesn't exist, python throws a `KeyError`.

</p>
</details>

In [ ]:
D = {'apple':100, 'watermelon':200,'orange':14}

#behavior of the get method:
print(D.get('apple'))
print(D.get('sausage'))

<details><summary><u>Solution.</u></summary>
<p>


```python
D = {'apple':100, 'watermelon':200,'orange':14}
def same_as_get(d,key):
    try:
        return d[key]
    except KeyError:
        return None
    
print(same_as_get(D,'apple'))
print(same_as_get(D,'sausage'))  
```

    
</p>
</details>

### 02 Conversion

Write a function that takes a string as input, if possible, converts it to an integer using the `int()` function and returns this value, if the conversion is not possible, it returns `None`.


<details><summary><u>Hint.</u></summary>
<p>

Run `int("hello")` and see what error is thrown. Or you can try to find the right error code [here](https://docs.python.org/3/tutorial/errors.html).

    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
def IntConv(x):
    try:
        return int(x)
    except ValueError:
        return None
    
IntConv("2.") 
```

    
</p>
</details>

### 03 Indexing

Write a function that takes one variable `x` as input and returns an element of it indexed by 2 (i.e., `x[2]`) if possible. If not possible return `None` and print out:
* "Index is out of range, buddy." if the index is out of range,
* "Please pay more attention to your variable types." if `x` is not indexable,
* "Something is not working..." in any other case.

Note that you can handle multiple error types within the same `try` statement simply by adding multiple `except` statements:
```python
try:
    some code
except Error1:
    some code
except Error2:
    some code
```

Write cases to test all error messages.

<details><summary><u>Hint.</u></summary>
<p>


Try out these `x`s to see the possible error codes:
```python
x=0
x[2]
    
x=[1,2]
x[2]
    
x={'hello':33}
x[2]
```
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
def index2(x):
    try:
        return x[2]
    except IndexError:
        print("Index is out of range, buddy.")
    except TypeError:
        print("Please pay more attention to your variable types.")
    except:
        print("Something is not working...")
        
    
index2({1:[3]}) 
```

    
</p>
</details>

## Exercises -- JSON/webapi

### 04 JSON

Convert the string `S` into a JSON object `D` and do the following:
* Try pretty printing `D` (printing in a more human readble way) by converting it back to a string using the argument `indent`, check the documentation for details.

<details><summary><u>Hint.</u></summary>
<p>

To convert back to a string use `json.dumps()`.
    
</p>
</details>

In [ ]:
S = '{"movies": {"Repo Man": {"actors": ["Emilio Estevez", "Harry Dean Stanton", "Zander Schloss"], "imdb_rating": 6.9, "year": 1984, "director": "Alex Cox"}, "Human Highway": {"actors": ["Neil Young", "Mark Mothersbaugh", "Pegi Young"], "imdb_rating": 6.0, "year": 1982, "director": "Neil Young"}, "Mighty Ducks": {"actors": ["Heidi Kling", "Emilio Estevez"], "imdb_rating": 6.6, "year": 1992, "director": "Stephen Herek"}}, "band_membership": {"Emilio Estevez": [], "Harry Dean Stanton": ["Harry Dean Stanton & The Cheap Dates"], "Zander Schloss": ["Circle Jks", "The Weirdos"], "Neil Young": ["Crazy Horses", "Buffalo Springfield"], "Mark Mothersbaugh": ["DEVO"], "Pegi Young": ["Pegi Young and the Survivors"], "Heidi Kling": []}}'



<details><summary><u>Solution.</u></summary>
<p>


```python
D= json.loads(S)
print(json.dumps(D,indent=1))  
```

    
</p>
</details>

* Print out which movies had their directors also act in them.

<details><summary><u>Hint.</u></summary>
<p>

To test if `x` is contained in a list `L`:
```python
if x in L:
    ...
```
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
for movie,details in D['movies'].items():
    if details['director'] in details['actors']:
        print(movie)  
```

    
</p>
</details>

* Print out the title of the movies and the number of actors in the movie who are also musicians.

<details><summary><u>Hint.</u></summary>
<p>

You have to count the number of actors in `D['movies'][movie_title]['actors']` whose band membership list is not empty.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
#solution 1
for movie,details in D['movies'].items():
    n = 0
    for actor in details['actors']:
        if len(D['band_membership'][actor])>0:
            n += 1
    print(movie,n)
    
#solution 2
for movie,details in D['movies'].items():
    n = len([actor for actor in details['actors'] if D['band_membership'][actor]])
    print(movie,n) 
```

    
</p>
</details>

### Weather API

Web API services are typically marketed developers, who create apps or other software that rely on the data from the web APIs. Usually there is a free limited subscription that allows developers to test out an API before committing to it. We make use of of these subscriptions for our own eductaional purposes.

Register and get an app id (also known as the api key) from [weatherapi.com](https://www.weatherapi.com/). You need an email address to register. On an unrelated note there are a disposable email [services](https://www.google.com/search?&q=temporary%20email) that provide a temporary email that you can use once and then forget about. Sometimes these email addresses are not allowed by webservices, so if the first one does not work you can try the next one.

There is also an [api explorer](https://www.weatherapi.com/api-explorer.aspx) that let's you test and construct request urls on their website.

### 05 Current weather

This api has a wide range of functionality. For example the following request provides info about the current weather in Paris:
```python
url = "http://api.weatherapi.com/v1/current.json?key="+api_key+"&q=Paris"
```
Download the current weather, convert it to a JSON object and print it out to explore the data it contains.

<details><summary><u>Hint.</u></summary>
<p>

Use the same steps we used for `openexchangerates.org`.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
url="http://api.weatherapi.com/v1/current.json?key="+api_key+"&q=Paris"
result = urllib.request.urlopen(url)
text = result.read()
current_weather = json.loads(str(text,"utf-8"))
print(json.dumps(current_weather,indent=1))
```

    
</p>
</details>

### 06 Rain --  Discussion exercise

Write a function that takes your location as input and prints out "yes" if it is raining outside and "no" if it is not.

Take a look at the [api documentation](https://www.weatherapi.com/docs/)!

**Share your solution on Slack. We will discuss the problem during class together.**

### 07 Forecast

Someone I know moved from Davis, California to Budapest for a new job. Did this person make the right decision? Get tomorrows hourly weather forecast for Davis, California and Budapest and plot the predicted temperature as a function of hour of the day.

<details><summary><u>Hint.</u></summary>
<p>

Request the forecast for two days, the second day in the list will be tomorrow's forecast:
```python
weather['forecast']['forecastday'][1]
```
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>


```python
for loc in ['Davis','Budapest']:
    url="http://api.weatherapi.com/v1/forecast.json?key="+api_key+"&q="+loc+"&days=2"
    result = urllib.request.urlopen(url)
    text = result.read()
    weather = json.loads(str(text,"utf-8"))
    temps = [hw['temp_c'] for hw in weather['forecast']['forecastday'][1]['hour']]

    plt.title(weather['forecast']['forecastday'][1]['date'])
    plt.plot(temps,'o-',label=loc)
    
plt.xlabel('hours')
plt.ylabel('temperature[C]')
plt.legend()
```

    
</p>
</details>

## Exercises -- XML/HTML

As part of the exercises you will have to extract some data from a file or website using BeautifulSoup and typically you will have to process this data (plot something, calculate a statistic, etc), you can do the second part multiple ways using built-in datatypes, numpy or pandas, the choice is yours. Similar to real applications, you might find that the simplest solution uses a function or trick that we did not cover in class, so look at the documentations and use the internet as needed.

### 08 Quiz XML

As an instructor of a course you can export the quizes in moodle as xml files. The file `python_quiz.xml` contains the questions from the quiz you did after the first class.

Load the file and parse it with BeautifulSoup.

In [ ]:
with open("python_quiz.xml","r",encoding='utf-8') as f:
    soup = BeautifulSoup(f,"xml")

<details><summary><u>Solution.</u></summary>
<p>
    
```python
with open("python_quiz.xml","r",encoding='utf-8') as f:
    soup = BeautifulSoup(f,"xml")
```
    
</p>
</details>

The questions are contained in `<question>` tags. Take the first one and figure out how to pretty print it, so you can examine its structure.

<details><summary><u>Hint</u></summary>
<p>

`tag.another_tag` returns the first `another_tag` inside `tag`, see class notebook. The `prettify()` tag method creates a formatted string.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
print(soup.question.prettify())
```
    
</p>
</details>

How many questions are there and what are the different types of question?

<details><summary><u>Hint</u></summary>
<p>

Iterate over all `<question>` tags.
    
One possible way to identify the different types is to collect all first occurances of the `type` attribute in a list. You can also check out the built-in data type `set`.
 
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
qtypes = []
for q in soup.find_all('question'):
    if q['type'] not in qtypes:
        qtypes.append(q['type'])
print("Number of questions:", len(soup.find_all('question')))
print("Question types:", qtypes)
```
    
</p>
</details>

How many questions contain the word "list" in the question itself (not the answers)?

<details><summary><u>Hint</u></summary>
<p>

The questions are encased in the `<questiontext>` tags.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
n_lists =0
for q in soup.find_all('question'):
    if "list" in q.questiontext.text:
        #print(q.questiontext.text)
        n_lists += 1
print(n_lists)
```
    
</p>
</details>

### 09 arXiv.org

[arXiv.org] is an open-access preprint repository containing almost 2,000,000 scietific papers (last time I checked). It originally started for Physics, but since has expanded to include math, computer science and other fields. It has an API that allows you to access the articles and their metadata. Use the URL below to retrive metadata of the first 1000 articles with titles containing the word "covid" in XML format. The information about the individual papers are stored in `<entry>` tags, print out the first one to see the available data.

<details><summary><u>Hint</u></summary>
<p>

Same as in the first exercise.
    
</p>
</details>

In [ ]:
url = 'http://export.arxiv.org/api/query?search_query=ti:"covid"&max_results=1000&sortBy=submittedDate&sortOrder=ascending'


<details><summary><u>Solution.</u></summary>
<p>
    
```python
data = urllib.request.urlopen(url)
soup = BeautifulSoup(data,"xml")
print(soup.entry.prettify())
```
    
</p>
</details>

Create a figure that shows the number of papers published in each month in the dataset!

There are many ways you can do this: you can create a dataframe, you can use lists. You can write your own code to count the occurences in a month or use `np.unique` or `collections.Counter`. Google away if you need help.

<details><summary><u>Hint</u></summary>
<p>

The date and time when the paper was published is in the `<published>` tag. Create a date object with the year and the month of publishing, but set the day to 1.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
YMs = [datetime.date(year=int(pub.string[:4]),month=int(pub.string[5:7]),day=1) for pub in soup.find_all('published')]
uniYMs, counts = np.unique(YMs, return_counts=True)
plt.plot(uniYMs,counts,'o-')
plt.ylabel("Number of covid papers")
plt.xticks(rotation=45);
``` 
</p>
</details>

You can also look at the newest 1000 paper changing "ascending" to "descending" in the url. (arXiv.org does not recommend api search requests that return more than 1000 entries. If you are interested in accessing more, you can stich together the results of multiple requests, or you can look into the bulk data api.)

Print out the title of the paper that has the most co-authors!

<details><summary><u>Hint</u></summary>
<p>

Count the number of `<author>` tags contained within the `<entry>` tags.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
max_auth=0

for entry in soup.find_all('entry'):
    auth = len(entry.find_all('author'))
    if auth>max_auth:
        max_auth = auth
        title = entry.title.string
print(title)
print('number of co-authors:', max_auth)
```
    
</p>
</details>

###  10 Planets

Download the table containing info about planets compiled for you by NASA from here: https://nssdc.gsfc.nasa.gov/planetary/factsheet/
Parse the table using BeautifulSoup and store the data in a pandas dataframe.

It's generally a good idea to download and process the data in separate cells, so you don't repeat the download unnecessarily.

<details><summary><u>Hint</u></summary>
<p>

* The table is contained in the `<table>` tag. The table rows are inside `<tr>` tags, the table cells are represented by `<td>` tags.
* The column names are in the first `<tr>` tag. Iterate over the rest of the `<tr>` tags to fetch the rows, the first `<td>` is the row index, the rest is the data.
* Collect the table data as a list of lists, convert it to a dataframe, and rename the colunms and rows.  
* You can use slices such as `soup.find_all(...)[1:]` to exclude tags as needed.
   
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
webpage = urllib.request.urlopen("https://nssdc.gsfc.nasa.gov/planetary/factsheet/")
soup = BeautifulSoup(webpage,"lxml")   
 
colnames = []
for td in soup.table.tr.find_all('td')[1:]:
    colnames.append(td.text.strip())
colnames

rownames = []
rows =[]
for tr in soup.table.find_all('tr')[1:-1]:
    rownames.append(tr.td.text.strip())
    rows.append([])
    for td in tr.find_all('td')[1:]:
        rows[-1].append(td.text.strip())

df=pd.DataFrame(rows)
df.columns=colnames
df.set_index(pd.Index(rownames),inplace=True)
df
```
    
</p>
</details>

For good measure, create a plot showing the relationship between mean temperature and distance from the Sun.

<details><summary><u>Hint</u></summary>
<p>

You can directly use matplotlib's `plt.plot()`. Or you can use pandas plot, but `df.plot(...)` plots columns not rows, one solition is to transpose your dataframe using `df.T`.
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
df.T.plot(kind='scatter',x="Distance from Sun (106 km)", y="Mean Temperature (C)")
```
    
</p>
</details>

### 11 Planets part 2

Now do the same thing using `pd.read_html()` (check out the documentation for details).

<details><summary><u>Hint</u></summary>
<p>

Note that `pd.read_html()` returns a list of dataframes, one for each table in the html file.
    
One option to drop the rows that are labelled `nan` with
```python
df = df.loc[df.index.dropna(),:] 
```
</p>
</details>

In [ ]:
import pandas as pd
df = pd.read_html("https://nssdc.gsfc.nasa.gov/planetary/factsheet/")[0]


<details><summary><u>Solution.</u></summary>
<p>
    
```python
df = pd.read_html("https://nssdc.gsfc.nasa.gov/planetary/factsheet/")[0]

df.set_index(0, inplace=True)
df.columns = df.iloc[0]
df = df.loc[df.index.dropna(),:]
df.index.name=None
df.columns.name = None

df
```
    
</p>
</details>

So why don't we always use pandas?
* To learn about html and its parsing
* Not all data from website comes from tables
* Not all html is clean enough and `pd.html_read()` might not work as expected
* etc

### 12 Christmas

Write a function `isitChristmas()` that prints no if it is not Christmas and yes if it is by scrapping the 
https://isitchristmas.today/ website.

**There is a catch:**<br>
When your browser sends a request to a server to download a page it also sends additional information about itself in the header of the request. Some websites block requests that do not have such information, and would not allow your script to access the contents. Luckily you can easily fake this by using
```python
#construct a request with the additional header
req = urllib.request.Request("https://isitchristmas.today/",headers={'User-Agent':'Mozilla/5.0'})
#send the request and download the site
webpage = urllib.request.urlopen(req)
```

To make sure your code works, you should test it for all possible scenarios! Therefore, run your solution on Christmas day too!

<details><summary><u>Hint</u></summary>
<p>

Visit the website with your browser and open the source code to see what tag is used to display the answer.
    
</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
def isitChristmas():
    req = urllib.request.Request("https://isitchristmas.today/",headers={'User-Agent':'Mozilla/5.0'})
    webpage = urllib.request.urlopen(req)
    soup = BeautifulSoup(webpage,"lxml") 
    
    
    if soup.html.body.h2.string=="No!":
        print("no")
    else:
        print("yes")
    
    return

isitChristmas()
```
    
</p>
</details>

Of course you could have done this a lot easier using datetime.

## Final problem

Solve either Part I, Option 1 or Option 2. In Part II, plan your final project.


### Part I -- Option 1

Write a function that returns two lists conataining the authors and the titles of the books on the New York Times best seller list.
* Download and parse the page from here: https://www.nytimes.com/books/best-sellers/combined-print-and-e-book-fiction/
* The book titles are encased in a tag with attribute itemprop="name"
* The authors are in a tag itemprop="author"





### Part I -- Option 2

Predicting exchange rates is a difficult problem, so it totally makes sense to get help from any possible source. Look for correlations between movements of celestial objects and currency exchange rates!

 `weatherapi.com` has an astronomy api that can provide the illuminated portion of the moon for each day. It ranges from 0% (new moon) to 100% (full moon).

* Download the moon illumination for the last 100 days.
* Download all exchange rates for the last 100 days.
* Find the currency that has the highest correlation with the moon phases (illuminated portion of the moon).
* Create a plot that compares the moon illumination time series to this exchange rate time series (use [this type](https://matplotlib.org/devdocs/gallery/subplots_axes_and_figures/two_scales.html) of plot).

Hint:
* You download the exchange rates as a dictionary; you can collect the dictionaries corresponding to each day in a list `L`; and this list can be easily converted to a dataframe with
```python
df_rates= pd.DataFrame(L),
```
where the rows represent the days and the columns the different exchange rates.


### Part II

Plan your final project! Check the requirement of the final project on moodle, and write a short paragraph describing your plans. Pick something that interests you and/or is useful for your research.